# 05 — Embeddings TerraMind (Etapa 4)

1. **Sub-AOI:** embeddings de todos os patch-trimestres do cubo (S2 e S2+S1), grade de tokens 14 × 14 e features espectrais. ~15 min em CPU.
2. **Conferência rápida:** a mudança de embedding ao longo do tempo nos patches de exemplo da inspeção.
3. **Gate do PLANO:** repetir o score de mudança por cosseno do first pass com TerraMind no lugar do
   AlphaEarth, nos mesmos pontos de teste (composições anuais). Se ficar muito abaixo (AUROC < 0,80 em
   "todas as anomalias"), investigar normalização, bandas e composição antes de seguir.
4. **AOIs completas:** rodam fora do notebook, de noite, com `stream_embeddings.py` (retomável). A última célula mostra o progresso.

Roda localmente (CPU). Saídas em `embeddings/` (fora do git).

In [ ]:
import sys, os
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    raise RuntimeError('Fluxo local desde 29/10 (ver "Limitações de infraestrutura" no PLANO). '
                       'No Colab, siga o setup do 03_cubo_sentinel e use /content/cache para os pixels.')
import json, time
import numpy as np
import pandas as pd
import config, utils, rotulos, dados, embeddings
config.make_dirs()
utils.set_seed(config.SEED)

run_dir = utils.new_run('etapa4_embeddings')
import ee
try:
    ee.Initialize(project=config.EE_PROJECT_ID)
except Exception:
    ee.Authenticate(); ee.Initialize(project=config.EE_PROJECT_ID)
model = embeddings.load_model()

## 1. Sub-AOI (a partir do cubo)

In [ ]:
t0 = time.time()
sub = embeddings.extract_cube(config.CUBE_DIR / config.CUBE_PATH_NAME, config.EMBEDDINGS_DIR / 'tapajos_sub.npz', model)
print(f'{(time.time() - t0) / 60:.1f} min |', {k: v.shape for k, v in sub.items() if k.startswith('z_')}, '| completo:', sub['done'].all())

## 2. Conferência: mudança do embedding nos patches de exemplo

Score = 1 − cosseno entre Z(t) e a média dos 4 trimestres anteriores (S2 + S1). Nos patches com evento, deve subir no trimestre do alerta (linha vertical); no normal, deve ficar baixo.

In [ ]:
import matplotlib.pyplot as plt
INK, MUTED, GRID = '#0b0b0b', '#52514e', '#e1e0d9'
CAT = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']      # slots categóricos na ordem fixa
plt.rcParams.update({'font.size': 9, 'axes.edgecolor': GRID, 'axes.labelcolor': MUTED, 'xtick.color': MUTED,
                     'ytick.color': MUTED, 'axes.spines.top': False, 'axes.spines.right': False})

import glob
insp = json.load(open(sorted(glob.glob(str(config.RESULTS_DIR / 'etapa3_inspecao' / '*' / 'resumo.json')))[-1], encoding='utf-8'))
Z = sub['z_s2s1'].astype(np.float32)
H = np.full_like(Z, np.nan)
for t in range(config.K_HIST, Z.shape[1]):
    H[:, t] = Z[:, t - config.K_HIST:t].mean(1)
cos = 1 - (Z * H).sum(-1) / (np.linalg.norm(Z, axis=-1) * np.linalg.norm(H, axis=-1))
pids, steps = list(sub['patch_id']), list(sub['steps'])

fig, ax = plt.subplots(figsize=(11, 3.4))
for c, (name, e) in zip(CAT, insp['exemplos'].items()):
    i = pids.index(e['patch'])
    ax.plot(range(len(steps)), cos[i], color=c, lw=2, label=name.split(' (')[0])
    if e['passo_evento']:
        ax.axvline(steps.index(e['passo_evento']), color=c, lw=1, ls='--')
ax.set_xticks(range(0, len(steps), 2), steps[::2], rotation=45)
ax.set_ylabel('1 − cos(Z(t), histórico)'); ax.grid(axis='y', color=GRID, lw=0.5)
ax.legend(frameon=False, ncol=4, loc='upper left'); ax.set_title('Mudança do embedding TerraMind (S2 + S1)', color=INK)
fig.tight_layout(); fig.savefig(run_dir / 'mudanca_exemplos.png', dpi=130); plt.show()

## 3. Gate: TerraMind × AlphaEarth nos pontos de teste do first pass

Para cada ponto de teste (671) e cada ano de T−2 a T+1: patch 224 × 224 centrado no ponto, composição **anual**
(mesma ideia do AlphaEarth anual), embeddings S2 e S2 + S1. Score = 1 − cosseno entre Z(T) e a média de Z(T−2), Z(T−1),
exatamente como o "AE cosine change (T)" do first pass. ~1 h de downloads; retomável.

Duas escalas: a **média do patch de 2,24 km** e a **célula de 320 m que contém o ponto** (a unidade adotada em 29/10).
O gate vale na célula. A média do patch fica como registro de por que a unidade mudou (os rótulos do first pass
são de um pixel, e a média de 2,24 km dilui o evento).

In [ ]:
fp = np.load(config.FIRSTPASS_DIR / 'data' / 'amazon_firstpass.npz', allow_pickle=True)
test = np.nonzero(fp['split'] == 'test')[0]
pairs = pd.DataFrame([{'pt': i, 'lon': fp['lon'][i], 'lat': fp['lat'][i], 'year': int(fp['T'][i]) + d, 'off': d}
                      for i in test for d in (-2, -1, 0, 1)])
g = embeddings.fetch_points(pairs, config.AOIS['tapajos']['epsg'], config.EMBEDDINGS_DIR / 'gate_firstpass.npz', model=model)

In [ ]:
import utils
STRATA_ANOM = [2, 3]
strat, block = fp['strat'][test], fp['block'][test]
rows = []
tasks = {'all anomalies vs all normals': (np.ones(len(test), bool), np.isin(strat, STRATA_ANOM)),
         'deforest vs stable forest': (np.isin(strat, [0, 2]), strat == 2),
         'deforest vs stable pasture': (np.isin(strat, [1, 2]), strat == 2),
         'mining vs all normals': (np.isin(strat, [0, 1, 3]), strat == 3)}
for cfg in embeddings.CONFIGS:
    for key, escala in [('cell', 'célula 320 m'), ('z', 'média patch 2,24 km')]:
        z = g[f'{key}_{cfg}'].reshape(len(test), 4, -1)        # anos T-2, T-1, T, T+1
        h = z[:, :2].mean(1)
        for off, name in [(2, 'T'), (3, 'T+1')]:
            a = z[:, off]
            sc = 1 - (a * h).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(h, axis=1))
            for task, (m, y) in tasks.items():
                y = y.astype(int)
                lo, hi = utils.block_bootstrap(block[m], y[m], sc[m], utils.auroc)
                rows.append({'task': task, 'score': f'TerraMind {cfg.upper()} {escala} cosine change ({name})',
                             'escala': escala, 'AUROC': utils.auroc(y[m], sc[m]), 'AUROC_lo': lo, 'AUROC_hi': hi,
                             'TPR_at_5FPR': utils.tpr_at_fpr(y[m], sc[m], 0.05)})
tm = pd.DataFrame(rows)
ref = pd.read_csv(config.FIRSTPASS_DIR / 'results' / 'metrics.csv')
ref = ref[ref['score'].isin(['AE cosine change (T)', 'AE cosine change (T+1)', 'AE kNN static, no time (T)',
                             'NDVI drop (T)', 'S2 band distance (T)'])]
gate = pd.concat([tm, ref.assign(escala='pixel 10 m (first pass)')[tm.columns]], ignore_index=True)
display(gate.pivot(index='score', columns='task', values='AUROC').round(3))
head = tm[(tm['task'] == 'all anomalies vs all normals') & tm['score'].str.endswith('(T)') & (tm['escala'] == 'célula 320 m')]
best = head['AUROC'].max()
print(f'Gate na célula de 320 m (AUROC >= 0,80 em todas as anomalias): melhor TerraMind = {best:.3f} ->',
      'PASSA' if best >= 0.80 else 'NÃO PASSA')
print('clear S2 médio nas composições anuais:', round(float(g['clear_s2'].mean()), 3))
gate.to_csv(run_dir / 'gate_firstpass.csv', index=False)

## 4. AOIs completas (fora do notebook)

Num terminal, a partir de `notebooks/gabriel` (deixe rodando de noite; Ctrl+C para parar, rode de novo para continuar):

```
..\..\venv\Scripts\python stream_embeddings.py tapajos
..\..\venv\Scripts\python stream_embeddings.py norte_ro
```

Blocos em ordem aleatória fixa: o que terminou é uma amostra aleatória de blocos. O `06_baselines` aceita AOIs parciais.

In [ ]:
for aoi in config.AOIS:
    p = config.EMBEDDINGS_DIR / f'{aoi}.npz'
    if p.exists():
        st = embeddings.load_store(p)
        print(f"{aoi}: {st['done'].mean():.1%} dos patch-trimestres | {st['done'].all(axis=1).sum()} patches completos")
    else:
        print(f'{aoi}: ainda não iniciado')
(run_dir / 'resumo.json').write_text(json.dumps({'gate': gate.to_dict('records'), 'gate_melhor_auroc': float(best)},
                                                indent=2), encoding='utf-8')